# Email Spam Detection with ML

## Introduction
Email spam detection is one of the most common applications of Natural Language Processing (NLP) and machine learning. Spam messages often contain unsolicited advertisements, phishing attempts, or fraudulent content, making automatic detection essential for improving communication security and user experience.

The objective of this project is to develop a binary classification model capable of distinguishing spam messages from ham (legitimate) messages. The SMS Spam Collection dataset is first cleaned and explored through exploratory data analysis (EDA) to understand its characteristics. The text messages are then preprocessed by removing unnecessary characters and common words before being converted into numerical features using the Term Frequency–Inverse Document Frequency (TF-IDF) technique.

Finally, two machine learning algorithms, Multinomial Naive Bayes and Logistic Regression, are trained and evaluated using performance metrics such as accuracy, precision, recall, F1-score, ROC curves, and AUC scores. The performance of both models is compared to determine the most effective approach for spam detection.

### Import Libraries

In [134]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


### Load dataset

In [135]:
df = pd.read_csv("spam.csv", encoding="latin-1")

# Data Cleaning and Preprocessing

### Dataset Exploration

Before preprocessing, the dataset is explored to understand its structure, dimensions, and data types. Inspecting the first few records, the number of rows and columns, and the available features helps identify any issues that may require cleaning before model development 

In [136]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [137]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [138]:
df.describe(include="all")

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
count,5572,5572,50,12,6
unique,2,5169,43,10,5
top,ham,"Sorry, I'll call later","bt not his girlfrnd... G o o d n i g h t . . .@""","MK17 92H. 450Ppw 16""","GNT:-)"""
freq,4825,30,3,2,2


In [139]:
df.shape

(5572, 5)

In [140]:
df.sample(5)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
5530,ham,I think that tantrum's finished so yeah I'll b...,NaN,NaN,NaN
4388,ham,"K I'm ready, &lt;#&gt; ?",NaN,NaN,NaN
977,ham,Dont hesitate. You know this is the second tim...,NaN,NaN,NaN
4179,ham,"swhrt how u dey,hope ur ok, tot about u 2day.l...",NaN,NaN,NaN
1041,ham,I'm in class. Will holla later,NaN,NaN,NaN


### Inspecting Missing Values and Duplicates 



In [141]:
df.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [142]:
df.duplicated().sum()

np.int64(403)

### Handling missing values and duplicates 
The dataset was inspected for missing values. Three unnamed columns contained almost entirely missing values and did not contribute useful information to the spam detection task. These columns were removed to simplify the dataset while preserving the relevant information contained in the Label and Message columns. Duplicate messages were identified and removed from the dataset. Duplicate observations may bias the learning process by giving repeated messages greater influence during training. Removing duplicates helps improve the quality and reliability of the machine learning models.

In [143]:
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)

In [144]:
df.drop_duplicates(inplace=True)

### Renaming columns 
The original column names were renamed to more descriptive labels. The target variable was renamed to Label, representing whether a message is spam or ham, while the text feature was renamed to Message, improving the readability of the dataset.

In [145]:
df.rename(columns={
    'v1': 'Label',
    'v2': 'Message'
}, inplace=True)

In [146]:
df.shape

(5169, 2)

In [147]:
df.head()

,Label,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [148]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Label    5169 non-null   object
 1   Message  5169 non-null   object
dtypes: object(2)
memory usage: 121.1+ KB


# Exploratory Data Analysis 

## Class Distribution

The distribution of spam and ham messages was examined to understand the balance of the dataset. Since classification performance can be affected by class imbalance, analysing the number of messages in each category provides useful insight before model training.

In [149]:
df['Label'].value_counts()

Label
ham     4516
spam     653
Name: count, dtype: int64

In [150]:
Class Distribution 

SyntaxError: invalid syntax (3174259985.py, line 1)

In [ ]:
(df['Label'].value_counts(normalize=True) * 100).round(2)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Label', data=df)

plt.title("Spam vs Ham Message Count")
plt.xlabel("Message Type")
plt.ylabel("Count")

plt.show()

Above graph shows the class distribution between spam and ham messages. We can see that the dataset contains more ham messages fewer spams.

In [ ]:
df['Label'].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    figsize=(6,6)
)

plt.ylabel("")
plt.title("Distribution of Spam and Ham Messages")

plt.show()

As observed from the above graph, the pie chart too shows the distribution between the two classes. 87.4% messages in the datset constitute ham whilst 12.6% are spam. 

## Feature engineering

In this project, a new feature called Message_Length was derived by calculating the number of characters in each message. This feature was created to investigate whether spam messages tend to be longer or shorter than legitimate (ham) messages and to provide additional insight during exploratory data analysis. This analysis helps determine whether message length may be a useful characteristic for distinguishing spam from legitimate messages.

In [ ]:
df['Message_Length'] = df['Message'].apply(len)

In [ ]:
df.head()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df['Message_Length'], bins=40)

plt.title("Distribution of Message Length")
plt.xlabel("Number of Characters")
plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.boxplot(x='Label', y='Message_Length', data=df)

plt.title("Message Length by Class")
plt.xlabel("Message Type")
plt.ylabel("Number of Characters")

plt.show()

The box plot shows that spam messages generally contain more characters than ham messages, as indicated by the higher median message length. Spam messages also exhibit less variation in length, whereas ham messages display a wider spread and numerous long-message outliers. Although message length alone cannot completely separate spam from ham due to some overlap between the two classes, it provides useful information for distinguishing the two message types. There is indeed a noticeable difference between message length of the thwo classes. 

# NLP TEXT PREPROCESSING

### Import NLP Libraries 

In [ ]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

### Downloading NLTK stopwords

In [ ]:
nltk.download('stopwords')

## Text Preprocessing

The message text was preprocessed to remove unnecessary information while preserving meaningful words. The preprocessing pipeline included converting all text to lowercase, removing punctuation and special characters, eliminating common stopwords, and applying stemming to reduce words to their root forms. These steps help reduce noise, decrease vocabulary size, and improve the quality of features used by the machine learning models.

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [ ]:
def preprocess_text(text):
    
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]
    return " ".join(words)

In [ ]:
df['Processed_Message'] = df['Message'].apply(preprocess_text)

In [ ]:
df[['Message', 'Processed_Message']].head()

## Feature Extraction

TF-IDF Vectorization:
Because of the fact tha Machine Learning models cannot process string data types, we therefore convert these processed text messages into numerical features that machine learning models can use.
Term Frequency–Inverse Document Frequency (TF-IDF) is a feature extraction technique used in Natural Language Processing (NLP). It converts text into numerical vectors by measuring how important a word is within a document relative to the entire dataset. Frequently occurring words in a specific message receive higher importance, while words that appear in almost every message receive lower importance. This enables machine learning algorithms to process textual data effectively.

### Import Libraries 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

### Convert text into numericals 

In [ ]:
tfidf = TfidfVectorizer()

X = tfidf.fit_transform(df['Processed_Message'])

y = df['Label']

In [ ]:
print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

### Train Test Split 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print("Training Set:", X_train.shape)
print("Testing Set :", X_test.shape)

# ML Model Training 

### Import ML libraries

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## Multinomial Naive Bayes

Multinomial Naive Bayes is a probabilistic classification algorithm widely used for text classification tasks such as spam detection. It assumes that words occur independently within a document and predicts the probability that a message belongs to a particular class based on the frequency of words. Due to its simplicity and efficiency, it serves as a strong baseline model for NLP applications.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

In [ ]:
y_pred_nb = nb_model.predict(X_test)

### Evaluation metrics

In [ ]:
accuracy = accuracy_score(y_test, y_pred_nb)
print("Accuracy:", accuracy)

In [ ]:
precision = precision_score(y_test, y_pred_nb, pos_label='spam')
print("Precision:", precision)

In [ ]:
recall = recall_score(y_test, y_pred_nb, pos_label='spam')
print("Recall:", recall)

Recall **(72.4%)** is relatively low as compared to precision **(100%)**. Recall is particularly important in spam detection because it measures the model's ability to correctly identify spam messages. A low recall means that many spam messages are incorrectly classified as legitimate (ham), allowing unwanted or potentially harmful messages to reach users. Therefore, improving recall helps ensure that fewer spam messages are missed.

In [ ]:
f1 = f1_score(y_test, y_pred_nb, pos_label='spam')
print("F1 Score:", f1)

In [ ]:
print(classification_report(y_test, y_pred_nb))

In [ ]:
cm = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Ham','Spam'],
    yticklabels=['Ham','Spam']
)

plt.title("Confusion Matrix - Naive Bayes")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.show()

The Multinomial Naive Bayes model was evaluated using accuracy, precision, recall, and F1-score. The confusion matrix illustrates the number of correctly and incorrectly classified spam and ham messages. High precision indicates that messages classified as spam are usually spam, while high recall indicates that most spam messages are successfully detected. The F1-score provides a balanced measure of both precision and recall.

Accuracy **(96.13%)** indicates that the model correctly classified the majority of SMS messages.
Precision **(100%)** means that every message predicted as spam was actually spam. In other words, the model made no false spam alarms.
Recall **(72.41%)** indicates that the model detected approximately **72%** of all spam messages, while the remaining spam messages were incorrectly classified as ham.
F1-score **(84%**) reflects a good balance between precision and recall, demonstrating overall strong classification performance.

### Plotting the ROC/AUC 

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
y_test_binary = y_test.map({'ham': 0, 'spam': 1})

In [ ]:
y_prob_nb = nb_model.predict_proba(X_test)[:, 1]

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test_binary, y_prob_nb)

auc_score = roc_auc_score(y_test_binary, y_prob_nb)

In [ ]:
plt.figure(figsize=(7,6))

plt.plot(fpr, tpr,
         label=f'Naive Bayes (AUC = {auc_score:.3f})',
         linewidth=2)

plt.plot([0,1], [0,1], 'k--', label='Random Classifier')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Multinomial Naive Bayes')

plt.legend(loc='lower right')

plt.show()

ROC Curve Interpretation:

The ROC curve lies close to the upper-left corner of the graph, indicating excellent classification performance. The model achieved an Area Under the Curve (AUC) score of 0.983, which is very close to the maximum value of 1.0. This demonstrates that the Multinomial Naive Bayes classifier is highly effective at distinguishing spam messages from legitimate (ham) messages across different decision thresholds.

## Logistic Regression Model

Logistic Regression is a supervised machine learning algorithm used for binary classification problems. It estimates the probability that an observation belongs to a particular class using the logistic (sigmoid) function. In this project, Logistic Regression is used to classify SMS messages as either spam or ham based on their TF-IDF feature representations.

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_model = LogisticRegression(random_state=42)

lr_model.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr_model.predict(X_test)

### Evaluation metrics

In [ ]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Accuracy:", accuracy_lr)

In [ ]:
precision_lr = precision_score(
    y_test,
    y_pred_lr,
    pos_label='spam'
)

print("Precision:", precision_lr)

In [ ]:
recall_lr = recall_score(
    y_test,
    y_pred_lr,
    pos_label='spam'
)

print("Recall:", recall_lr)

In [ ]:
f1_lr = f1_score(
    y_test,
    y_pred_lr,
    pos_label='spam'
)

print("F1 Score:", f1_lr)

In [ ]:
print(classification_report(y_test, y_pred_lr))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm_lr,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=['Ham','Spam'],
    yticklabels=['Ham','Spam']
)

plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.show()

### PLotting the AUC/ROC

In [ ]:
y_prob_lr = lr_model.predict_proba(X_test)[:,1]

fpr_lr, tpr_lr, thresholds_lr = roc_curve(
    y_test_binary,
    y_prob_lr
)

auc_lr = roc_auc_score(
    y_test_binary,
    y_prob_lr
)

plt.figure(figsize=(7,6))

plt.plot(
    fpr_lr,
    tpr_lr,
    label=f'Logistic Regression (AUC = {auc_lr:.3f})',
    linewidth=2
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")

plt.legend()

plt.show()

## Model comparison 

Given the different evaluation metrics for each kodel, we can easily compare the two to choose which model perfomance best in detecting spam emails from non spam emails.

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

naive_bayes = [
    accuracy,
    precision,
    recall,
    f1
]

logistic_regression = [
    accuracy_lr,
    precision_lr,
    recall_lr,
    f1_lr
]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(8,6))

plt.bar(x - width/2, naive_bayes, width,
        label='Naive Bayes')

plt.bar(x + width/2, logistic_regression, width,
        label='Logistic Regression')

plt.xticks(x, metrics)
plt.ylim(0,1.1)

plt.ylabel('Score')
plt.title('Performance Comparison of Classification Models')

plt.legend()

plt.show()

The above barchart shows the comaprison of every metrics between the two ML models, we can see that overally the Naive Bayes model performas slightly better than the Logistic Regression model.

In [ ]:
plt.figure(figsize=(7,6))

plt.plot(fpr, tpr,
         label=f'Naive Bayes (AUC = {auc_score:.3f})',
         linewidth=2)

plt.plot(fpr_lr, tpr_lr,
         label=f'Logistic Regression (AUC = {auc_lr:.3f})',
         linewidth=2)

plt.plot([0,1],[0,1],
         'k--',
         label='Random Classifier')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')

plt.legend(loc='lower right')

plt.show()

Both models achieved excellent classification performance, with AUC values above 0.98, indicating a strong ability to distinguish spam messages from legitimate messages. However, Multinomial Naive Bayes outperformed Logistic Regression in terms of accuracy, precision, recall, and F1-score. While Logistic Regression achieved a marginally higher AUC (0.984), its lower recall indicates that it failed to identify more spam messages than Naive Bayes. Therefore, Multinomial Naive Bayes was selected as the better-performing model for this spam detection task. 

Naive Bayes: Accuracy = 96.1%, Recall = 72%. Logistic Regression: AUC = 0.984 (slightly higher), but Recall = 66% This shows why AUC is not the only metric to be considered when selecting a model. Since spam detection aims to catch as many spam messages as possible, recall is especially important. Even though Logistic Regression has a tiny advantage in AUC, Naive Bayes detects more spam messages and also achieves higher overall accuracy.

## Conclusion:

This project successfully developed and evaluated machine learning models for Email spam detection using Natural Language Processing (NLP). The text messages were preprocessed through text cleaning techniques, transformed into numerical features using the TF-IDF vectorization method, and classified using Multinomial Naive Bayes and Logistic Regression.

Both models achieved strong classification performance, with AUC values greater than 0.98, indicating an excellent ability to distinguish between spam and legitimate messages. However, Multinomial Naive Bayes produced the best overall performance, achieving an accuracy of 96.13%, perfect precision (100%), a recall of 72.41%, and an F1-score of 0.84. Although Logistic Regression achieved a slightly higher AUC (0.984), it produced lower accuracy, recall, and F1-score than Naive Bayes.

These results indicate that Multinomial Naive Bayes is the more suitable model for this dataset. The project also demonstrates that combining effective text preprocessing with TF-IDF feature extraction can produce highly accurate spam detection models. Such systems can be applied to automatically filter unwanted messages and improve the efficiency of email and messaging platforms.